# 12 · LegalBench — the evaluation suite beside the pipeline

LegalBench is a **second lens** on model quality: it does not sit inside the
13-node graph. Two task families run against locally mirrored CUAD corpora
(or, here, a miniature fixture so the notebook is network-free).

**What you'll see:** the live task registry, a mock `contract_qa` run, a mock
`family_classification` run, and how that relates to the Hub dataset
`Lucius-Morningstar/legalbench-full` (currently a stub viewer).

**Honesty label:** `run_mini` uses `legalbench.runner.run_task(..., mock=True)`
on a 2-contract fixture (`notebooks/fixtures/legalbench/`). Scores are
`mock/mock-legalbench`, not a real model. The full 20,910-question corpus
lives at `data/cuad/` after `scripts/fetch_full_cuad.py`. OFFLINE.


## Setup


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))

import legalbench_lab as lb
import huggingface_lab as hf
print("mini CUAD:", lb.MINI_CUAD.exists(), lb.MINI_CUAD)
print("mini contracts dir:", lb.MINI_CONTRACTS.exists())


mini CUAD: True /workspace/notebooks/fixtures/legalbench/cuad_mini.json
mini contracts dir: True


## Task registry (live `legalbench.tasks`)


In [2]:
for t in lb.task_table():
    print(f"{t['id']:24s} kind={t['kind']}")
    print(f"{'':24s} prompt={t['prompt_version']}  classes={t['n_classes']}  e.g. {t['classes_head']}")


contract_qa              kind=legalbench_binary_answer
                         prompt=legalbench_contract_qa_v1  classes=2  e.g. ['yes', 'no']
family_classification    kind=legalbench_multiclass_classification
                         prompt=legalbench_family_classification_v1  classes=26  e.g. ['affiliate', 'agency', 'collaboration', 'co_branding', 'consulting', 'development', 'distributor', 'endorsement']


## Mock contract_qa on the miniature corpus

6 (contract × clause-category) questions, deterministic fake model, no
Langfuse. Scoring is local and never LLM-graded.


In [3]:
qa = lb.run_mini("contract_qa", n=6, seed=1)
print("task:", qa["task"], "model:", qa["model"], "n:", qa["n"])
print("scores:", {k: qa["scores"][k] for k in list(qa["scores"])[:8]})
print("honesty:", qa["honesty"])
print()
for i, row in enumerate(qa["rows"], 1):
    mark = "ok" if row["correct"] else "miss"
    print(f"  {i}. {mark:4s} expected={row['expected']!r:5s} predicted={row['predicted']!r}")


2026-08-25 02:13:31 [debug    ] score_configs_validated        count=37 registry=llm-dojo-scoring


task: contract_qa model: mock/mock-legalbench n: 6
scores: {'accuracy': 0.8333, 'macro_category_accuracy': 0.8, 'yes_f1': 0.8, 'n_questions': 6, 'n_yes': 3, 'n_no': 3, 'n_error': 0, 'confidence_mean': 0.5}
honesty: mock/mock-legalbench on a 2-contract fixture — not the full CUAD corpus.

  1. ok   expected='no'  predicted='no'
  2. ok   expected='yes' predicted='yes'
  3. ok   expected='yes' predicted='yes'
  4. ok   expected='no'  predicted='no'
  5. ok   expected='no'  predicted='no'
  6. miss expected='yes' predicted='no'


## Mock family_classification (25 CUAD families + other)


In [4]:
fam = lb.run_mini("family_classification", n=2, seed=3)
print("task:", fam["task"], "n:", fam["n"])
print("scores:", {k: fam["scores"].get(k) for k in ("accuracy", "accuracy_equiv", "macro_f1")})
for i, row in enumerate(fam["rows"], 1):
    print(f"  {i}. expected={row['expected']!r:16s} predicted={row['predicted']!r} correct={row['correct']}")


task: family_classification n: 2
scores: {'accuracy': 0.0, 'accuracy_equiv': 0.0, 'macro_f1': 0.0}
  1. expected='distributor'    predicted='other' correct=False
  2. expected='license'        predicted='strategic_alliance' correct=False


## Hub `legalbench-full` is not this suite

The published dataset currently has placeholder viewer rows. The real eval
is `python -m legalbench.cli --task contract_qa --n 30 --mock` against
`data/cuad/`.


In [5]:
stub = hf.preview("Lucius-Morningstar/legalbench-full")
print("hub rows:", len(stub.get("rows") or []), "features:", stub.get("features"))
print("first row:", (stub.get("rows") or [{}])[0])
print()
print("Use legalbench.cli for a real (still mockable) run; this notebook stays on the fixture.")


hub rows: 8 features: ['text']
first row: {'text': '{{text}}'}

Use legalbench.cli for a real (still mockable) run; this notebook stays on the fixture.


## Where to go next

- **11 huggingface_corpora** — CUAD + DE-SynPUF + Enron on the Hub
- **01 happy_path_run** — the pipeline this eval sits *beside*
- **08 observability_traces** — how a LegalBench run would look in Langfuse
